In [14]:
import whisper
from deepface import DeepFace
import tensorflow as tf
from tensorflow.keras import mixed_precision
import cv2
import os
import sys
import time
from IPython.display import clear_output

In [15]:
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0:All, 1:Filter Info, 2:Filter Warning, 3:Filter Error
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' # Mematikan peringatan oneDNN
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Gunakan GPU pertama (jika ada). Set ke "" untuk CPU saja.
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# print(tf.__version__)
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

print("Is GPU available?", tf.test.is_gpu_available())
print("Device Name:", tf.test.gpu_device_name())

GPU Memory Growth Enabled
Num GPUs Available:  1
Is GPU available? True
Device Name: /device:GPU:0


I0000 00:00:1776848140.162861    8076 gpu_device.cc:2043] Created device /device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1776848140.170822    8076 gpu_device.cc:2043] Created device /device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [17]:
def process_video_with_overlay(input_path, output_path, detector_backend='opencv', frame_step=3):
   # 1. Buka Video Source
   cap = cv2.VideoCapture(input_path)
   if not cap.isOpened():
      print(f"Error: Video {input_path} tidak ditemukan.")
      return

   # Properti Video (Gunakan ukuran target resize untuk writer!)
   # target_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) # Resize ke setengah untuk kecepatan
   # target_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
   target_width = 640
   target_height = 360
   fps = cap.get(cv2.CAP_PROP_FPS)
   total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
   
   # Siapkan Video Writer dengan ukuran yang sama dengan small_frame
   fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
   out = cv2.VideoWriter(output_path, fourcc, fps, (target_width, target_height))

   current_frame = 0
   start_time = time.time()
   dominant_emotion = "Mencari..."
   rect_coords = None

   print(f"Memproses: {os.path.basename(input_path)}")

   while cap.isOpened():
      ret, frame = cap.read()
      
      # CRITICAL FIX: Cek ret SEBELUM resize
      if not ret or frame is None: 
         break
      
      current_frame += 1
      
      # Resize frame segera setelah dibaca
      small_frame = cv2.resize(frame, (target_width, target_height))
      
      # Update analisis setiap N frame (Gunakan frame_step agar lebih fleksibel)
      if current_frame % frame_step == 0:
         try:
            # Real-time Jupyter progress update
            elapsed = time.time() - start_time
            minutes = int(elapsed // 60)
            seconds = int(elapsed % 60)
            milliseconds = int((elapsed - int(elapsed)) * 1000)
            time_elapsed = f"{minutes:02d}:{seconds:02d}.{milliseconds:02d}"
            progress = (current_frame / total_frames) * 100
            
            # Menggunakan clear_output agar tidak memenuhi layar Jupyter
            clear_output(wait=True)
            print(f"Progress: {current_frame}/{total_frames} ({progress:.2f}%) | Time: {time_elapsed}")

            # Analisis emosi
            analysis = DeepFace.analyze(
               small_frame, 
               actions=['emotion'], 
               enforce_detection=False, 
               detector_backend=detector_backend
               # detector_backend='ssd' # Lebih cepat dari opencv, tapi pastikan sudah terinstall (pip install deepface[ssd])
            )
            
            # DeepFace return list, ambil item pertama
            dominant_emotion = analysis[0]['dominant_emotion']
            rect_coords = analysis[0]['region']
               
         except Exception as e:
            # Jangan biarkan error satu frame menghentikan seluruh video
            print(f"Error pada frame {current_frame}: {e}")
            pass

      # Gambar ulang hasil terakhir pada frame saat ini
      if rect_coords:
         x, y, w, h = rect_coords['x'], rect_coords['y'], rect_coords['w'], rect_coords['h']
         cv2.rectangle(small_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
         cv2.putText(small_frame, f"{dominant_emotion.upper()}", (x, y - 10), 
         cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

      # Simpan frame yang sudah di-resize
      out.write(small_frame)

   # Clean up
   cap.release()
   out.release()
   print(f"\nSelesai! Tersimpan di: {output_path}")

In [ ]:
# for a  realtime analysis using camera
def start_realtime_emotion():
   cap = cv2.VideoCapture(0, cv2.CAP_V4L2) # use CAP_V4L2 for 

   cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
   cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

   if not cap.isOpened():
      print("Error: Kamera tidak terdeteksi. Jika di WSL2, pastikan usbipd sudah terhubung.")
      return

   print("Tekan 'q' untuk berhenti...")

   while True:
      ret, frame = cap.read()
      if not ret:
         break

      try:
         # 2. Analisis Wajah (Hanya emosi agar FPS tetap tinggi)
         # detector_backend 'opencv' adalah yang tercepat untuk real-time
         results = DeepFace.analyze(frame, 
                                    actions=['emotion'], 
                                    enforce_detection=False, 
                                    detector_backend='opencv')

         # 3. Gambar hasil analisis ke layar
         for face in results:
               x, y, w, h = face['region']['x'], face['region']['y'], face['region']['w'], face['region']['h']
               emotion = face['dominant_emotion']

               # Gambar kotak di wajah
               cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
               
               # Tulis teks emosi
               cv2.putText(frame, f"Emosi: {emotion.upper()}", (x, y - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

      except Exception:
         # Lewati jika wajah tidak terdeteksi dengan jelas di frame tersebut
         pass

      # 4. Tampilkan jendela video
      cv2.imshow('AIView Real-time Analysis', frame)

      # Berhenti jika menekan tombol 'q'
      if cv2.waitKey(1) & 0xFF == ord('q'):
         break

   # Bersihkan resource
   cap.release()
   cv2.destroyAllWindows()

start_realtime_emotion()

Extract emotion perframe into Dataframe

In [26]:
def extractEmotionFromVideo(input_path, frame_step=3, detector_backend='opencv'):
   cap = cv2.VideoCapture(input_path)
   if not cap.isOpened():
      print(f"Error: Video {input_path} tidak ditemukan.")
      return

   total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
   current_frame = 0
   emotions_over_time = []

   while cap.isOpened():
      ret, frame = cap.read()
      if not ret:
         break

      current_frame += 1

      if current_frame % frame_step == 0:
         try:
            analysis = DeepFace.analyze(frame, 
                                       actions=['emotion'], 
                                       enforce_detection=False, 
                                       detector_backend=detector_backend)
            # dominant_emotion = analysis[0]['dominant_emotion']
            emotions_over_time.append((current_frame, analysis))
         except Exception:
            emotions_over_time.append((current_frame, "Tidak terdeteksi"))

   cap.release()
   return emotions_over_time

In [20]:
# DATASET folder location
dataset_folder = "/mnt/c/DATA D/aa-kuliah/skripsi/DATASET"

In [ ]:
"""
   For a real-time analysis using camera, you can call the function start_realtime_emotion() at the end of this script.
"""

video_file = "videos/tips_interview_1080p.mp4" # Ganti dengan nama file video Anda

if os.path.exists(video_file):
   # If you want to save the video with emotion overlay, uncomment the lines below and specify the output path.
   # output_video = "/mnt/c/Users/zero/Documents/result.mp4"
   output_video = "/mnt/c/DATA D/aa-kuliah/skripsi/AIVIEW/output/640x360.mp4"
   process_video_with_overlay(video_file, output_video)
   
else:
   print(f"Error: File '{video_file}' tidak ditemukan.")

In [ ]:
# Batch processing for dataset videos (limit number of videos for demo)

limit = 20 # Batasi jumlah video yang diproses untuk demo
count = 0

frame_step = 5
detector_backend = 'opencv' # Pilihan: 'opencv', 'ssd'

for root, dirs, files in os.walk(f"{dataset_folder}/first-impressions/train/"):
   print(f"Video: {count + 1} / {limit}")
   for file in files:
      if file.endswith(".mp4"):
         video_path = os.path.join(root, file)
         print(f"Processing: {video_path}")
         output_video = video_path.replace("train", f"output/{detector_backend}/skip_{frame_step}_frames").replace(".mp4", "_processed.mp4")

         if not os.path.exists(root.replace("train", f"output/{detector_backend}/skip_{frame_step}_frames")):
            os.makedirs(root.replace("train", f"output/{detector_backend}/skip_{frame_step}_frames"))

         process_video_with_overlay(video_path, output_video, detector_backend=detector_backend, frame_step=frame_step)
         count += 1
         if count >= limit:
            break
   if count >= limit:
      break

In [27]:
# Extract emotions from a single video (for demo)
video_file = os.path.join(dataset_folder, "first-impressions/train/d4cPiUXpGbc.005.mp4") # Ganti dengan nama file video Anda

if os.path.exists(video_file):
   emotions = extractEmotionFromVideo(video_file, frame_step=3, detector_backend='opencv')
   # for frame_num, emotion in emotions:
   #    print(f"Frame {frame_num}: {emotion}")
else:
   print(f"Error: File '{video_file}' tidak ditemukan.")
   
emotions

[(3,
  [{'emotion': {'angry': np.float32(0.6191549),
     'disgust': np.float32(0.0047877855),
     'fear': np.float32(68.375824),
     'happy': np.float32(2.066342),
     'sad': np.float32(0.9931863),
     'surprise': np.float32(14.222605),
     'neutral': np.float32(13.718101)},
    'dominant_emotion': 'fear',
    'region': {'x': 494,
     'y': 4,
     'w': 415,
     'h': 415,
     'left_eye': (781, 145),
     'right_eye': (633, 151)},
    'face_confidence': 0.95}]),
 (6,
  [{'emotion': {'angry': np.float32(0.039481178),
     'disgust': np.float32(0.00029796),
     'fear': np.float32(34.78274),
     'happy': np.float32(0.23012023),
     'sad': np.float32(0.006583282),
     'surprise': np.float32(64.9244),
     'neutral': np.float32(0.016381435)},
    'dominant_emotion': 'surprise',
    'region': {'x': 500,
     'y': 12,
     'w': 408,
     'h': 408,
     'left_eye': (780, 145),
     'right_eye': (633, 152)},
    'face_confidence': 0.94}]),
 (9,
  [{'emotion': {'angry': np.float32(0.1

In [6]:
!ls /mnt/d/aa-kuliah/skripsi/Dataset/first-impressions/annotations/

annotation_test.pkl	   test-annotation-e.zip  val-annotation-e.zip
annotation_validation.pkl  train-annotation


In [46]:
import pandas as pd
import pickle
import os

dataset_root = '/mnt/d/aa-kuliah/skripsi/Dataset/first-impressions/'
train_root = os.path.join(dataset_root, "train")
train_annotation = os.path.join(dataset_root, 'annotations/train-annotation/annotation_training.pkl')

with open(train_annotation, "rb") as f:
   raw_data = pickle.load(f, encoding="latin1")
   
df = pd.DataFrame(raw_data)

df.sort_values(by=[df.columns[1], df.columns[2], df.columns[3], df.columns[4], df.columns[5]], inplace=True, ascending=False)
df

/tmp/ipykernel_1140/572300586.py:10: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  raw_data = pickle.load(f, encoding="latin1")


,extraversion,neuroticism,agreeableness,conscientiousness,interview,openness
jjFzwsootdM.000.mp4,0.757009,0.979167,0.879121,0.970874,0.906542,0.866667
7QeVs-mXmcI.003.mp4,0.850467,0.958333,0.912088,0.873786,0.971963,0.933333
84emxO86qa8.005.mp4,0.925234,0.958333,0.901099,0.912621,1.000000,0.922222
f8eU5Pc-y0g.004.mp4,0.654206,0.947917,0.813187,0.902913,0.813084,0.844444
OLFhKCgexRU.000.mp4,0.691589,0.947917,0.758242,0.970874,0.915888,0.900000
...,...,...,...,...,...,...
ZIk7N-xHvF4.002.mp4,0.084112,0.041667,0.153846,0.087379,0.065421,0.444444
69BopbFc34U.000.mp4,0.102804,0.031250,0.241758,0.135922,0.130841,0.188889
PHv6CzBIC5E.001.mp4,0.093458,0.031250,0.065934,0.184466,0.093458,0.000000
_kK9tGN883Y.000.mp4,0.000000,0.031250,0.054945,0.203883,0.158879,0.444444
